In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from google.colab import drive
import sys
import os

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 내 드라이브에 만든 바로가기 경로 설정
BASE_PATH = '/content/drive/MyDrive/Conference_2026'

# 기존 코드가 있는 Juhyeong 폴더와 영주 님의 개인 폴더 경로
JUHYEONG_PATH = os.path.join(BASE_PATH, 'Juhyeong')
YOUNGJU_PATH = os.path.join(BASE_PATH, 'Youngju')

# 3. 기존에 구현된 함수들을 가져다 쓰기 위해 시스템 경로에 추가
sys.path.append(os.path.join(JUHYEONG_PATH, 'scripts'))

# 4. 안전하게 새 파일을 만들기 위해 현재 작업 위치를 영주 님의 폴더로 이동
os.chdir(YOUNGJU_PATH)

print("경로 설정 완료! 현재 작업 위치:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
경로 설정 완료! 현재 작업 위치: /content/drive/.shortcut-targets-by-id/11nWT68DvNtPzI-UahXibXJheErrqufbu/Conference_2026/Youngju


In [8]:
import shutil

# 원본 스크립트 파일 경로
original_script = os.path.join(JUHYEONG_PATH, 'scripts', 'compute_ab_variance.py')

# 영주 님 폴더에 저장할 새 스크립트 파일 경로
new_script = os.path.join(YOUNGJU_PATH, 'compute_ab_variance_mad.py')

# 파일 복사 진행
shutil.copy(original_script, new_script)
print("스크립트 복사 완료! 이제 영주 님 폴더에서 파일을 수정할 수 있습니다.")

스크립트 복사 완료! 이제 영주 님 폴더에서 파일을 수정할 수 있습니다.


In [9]:
import os
import sys

# 1. 영주 님이 파일을 저장해둔 '5주차 Task' 폴더로 정확하게 경로를 수정합니다.
work_dir = "/content/drive/MyDrive/Conference_2026/Youngju/5주차 Task"
os.chdir(work_dir)

# 2. 현재 폴더를 파이썬이 가장 먼저 뒤지도록 첫 번째로 등록합니다.
if work_dir not in sys.path:
    sys.path.insert(0, work_dir)

# 3. 파일이 잘 있는지 눈으로 한 번 확인합니다.
if "compute_ab_variance_mad.py" in os.listdir():
    print("파일 확인 완료! 드디어 찾았네요. 이제 안심하고 실행합니다.")

    # 4. 실행할 때 넘겨줄 옵션들을 설정합니다.
    sys.argv = [
        "compute_ab_variance_mad.py",
        "--splits-dir", "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo",
        "--scores-dir", "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/scores_role4",
        "--out-dir", "/content/drive/MyDrive/Conference_2026/Youngju/signals_mad"
    ]

    # 이전에 꼬인 모듈 캐시가 남아있다면 깔끔하게 지워줍니다.
    if "compute_ab_variance_mad" in sys.modules:
        del sys.modules["compute_ab_variance_mad"]

    import compute_ab_variance_mad
    compute_ab_variance_mad.main()
else:
    print("여전히 파일이 안 보여요. 혹시 폴더 이름 띄어쓰기('5주차 Task')가 다를까요?")

파일 확인 완료! 드디어 찾았네요. 이제 안심하고 실행합니다.
[1/22] ames: 1,450행
[2/22] bbb_martins: 390행
[3/22] bioavailability_ma: 127행
[4/22] caco2_wang: 180행
[5/22] clearance_hepatocyte_az: 203행
[6/22] clearance_microsome_az: 220행
[7/22] cyp2c9_substrate_carbonmangels: 133행
[8/22] cyp2c9_veith: 2,409행
[9/22] cyp2d6_substrate_carbonmangels: 132행
[10/22] cyp2d6_veith: 2,616행
[11/22] cyp3a4_substrate_carbonmangels: 133행
[12/22] cyp3a4_veith: 2,460행
[13/22] dili: 94행
[14/22] half_life_obach: 132행
[15/22] herg: 127행
[16/22] hia_hou: 116행
[17/22] ld50_zhu: 1,468행
[18/22] lipophilicity_astrazeneca: 838행
[19/22] pgp_broccatelli: 242행
[20/22] ppbr_az: 358행
[21/22] solubility_aqsoldb: 1,915행
[22/22] vdss_lombardo: 222행

=== 축·모델별 신호 크기와 오차 상관 (test, 22종 중앙값) ===
       model           axis     신호평균     비영비율     상관중앙  상관양수
cb_augmented              A 0.048471 1.000000 0.109979    17
cb_augmented B1_protonation 0.010931 0.656681 0.074127    16
cb_augmented    B1_tautomer 0.024520 0.734326 0.042456    13
cb_augmented   

In [10]:
import pandas as pd
import numpy as np
import os
import sys

# 1. 경로 설정 및 함수 불러오기
juhyeong_scripts = "/content/drive/MyDrive/Conference_2026/Juhyeong/scripts"
if juhyeong_scripts not in sys.path:
    sys.path.append(juhyeong_scripts)

# analyze_heterogeneity.py에서 BH 통제 함수 가져오기
from analyze_heterogeneity import benjamini_hochberg

# 2. 골격 군집 단위 부트스트랩 함수 정의
def scaffold_bootstrap(df, scaffold_col='scaffold_group', n_iterations=100):
    """물성 안에서 골격 군집 단위로 재표집하는 함수"""
    bootstrapped_results = []
    unique_scaffolds = df[scaffold_col].unique()

    for i in range(n_iterations):
        # 군집 단위 복원 추출
        sampled_scaffolds = np.random.choice(unique_scaffolds, size=len(unique_scaffolds), replace=True)

        # 추출된 군집에 해당하는 데이터 병합
        sample_df = pd.concat([df[df[scaffold_col] == scaf] for scaf in sampled_scaffolds])
        bootstrapped_results.append(sample_df)

    return bootstrapped_results

print("통계 절차 함수 준비 완료! 내일 아침에 이어서 실제 데이터를 넣어 분석을 돌려보시면 됩니다.")

통계 절차 함수 준비 완료! 내일 아침에 이어서 실제 데이터를 넣어 분석을 돌려보시면 됩니다.


In [11]:
import inspect
from analyze_heterogeneity import benjamini_hochberg

# 함수 시그니처 및 문서 확인
print("함수 시그니처:", inspect.signature(benjamini_hochberg))
print("함수 구현 확인:\n", inspect.getsource(benjamini_hochberg))

함수 시그니처: (pvalues: 'np.ndarray') -> 'np.ndarray'
함수 구현 확인:
 def benjamini_hochberg(pvalues: np.ndarray) -> np.ndarray:
    order = np.argsort(pvalues)
    ranked = pvalues[order] * len(pvalues) / np.arange(1, len(pvalues) + 1)
    adjusted = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(adjusted)
    out[order] = np.clip(adjusted, 0, 1)
    return out



In [12]:
import pandas as pd
import numpy as np
from analyze_heterogeneity import benjamini_hochberg

np.random.seed(42)
mock_p_values = np.random.uniform(0, 0.1, 22)

# 1. 인자 없이 p-value만 전달하여 조정된 값을 반환받습니다.
pvals_corrected = benjamini_hochberg(mock_p_values)

# 2. alpha(0.05) 통과 여부는 반환된 값으로 외부에서 판단합니다.
rejected = pvals_corrected < 0.05

bh_results = pd.DataFrame({
    'original_p': mock_p_values,
    'corrected_p': pvals_corrected,
    'significant_after_BH': rejected
})

print("\n=== BH 다중비교 통제 결과 (상위 5개) ===")
print(bh_results.head())


=== BH 다중비교 통제 결과 (상위 5개) ===
   original_p  corrected_p  significant_after_BH
0    0.037454     0.074908                 False
1    0.095071     0.096991                 False
2    0.073199     0.089466                 False
3    0.059866     0.084130                 False
4    0.015602     0.057641                 False


In [13]:
import shutil
import os

# 1. 캡처 화면에 나타난 정확한 원본 파일 경로
original_path = "/content/drive/MyDrive/Conference_2026/Main/scripts_role4/run_preregistered_ablation.py"

# 2. 영주 님의 작업 폴더 경로 지정
target_dir = "/content/drive/MyDrive/Conference_2026/Youngju/5주차 Task"
os.makedirs(target_dir, exist_ok=True)
target_path = os.path.join(target_dir, "run_preregistered_ablation.py")

# 3. 원본은 그대로 둔 채 안전하게 복사
shutil.copy(original_path, target_path)
print("안전하게 복사 완료! 이제 영주 님 폴더에서 파일을 수정할 수 있습니다.")

안전하게 복사 완료! 이제 영주 님 폴더에서 파일을 수정할 수 있습니다.


In [17]:
%%writefile "/content/drive/MyDrive/Conference_2026/Youngju/5주차 Task/run_preregistered_ablation.py"
import pandas as pd, numpy as np, warnings
from pathlib import Path
from scipy.stats import rankdata
from sklearn.linear_model import Ridge
warnings.filterwarnings("ignore")

B = Path("/content/drive/MyDrive/Conference_2026")
EV = B / "Juhyeong/data/processed/scores_role4/evaluation"
SPLITS_DIR = B / "Juhyeong/data/processed/pipeline_yoonsoo"  # 군집 정보가 있는 경로

BASE_PRE = ["base__ad_knn__pct", "base__ad_density__pct", "base__conformal_cb__pct", "base__conformal_fp__pct"]
Bx = ["cond_B__fp_primary__std__pct", "cond_B__cb_augmented__std__pct"]

CFG = {
     "기준(AD+컨포멀)": BASE_PRE,
     "기준+B": BASE_PRE + Bx,
}

def naurc(s, e):
    f = lambda x: float(np.mean(np.cumsum(e[np.argsort(x, kind="stable")]) / np.arange(1, len(e) + 1)))
    o, r = f(e), float(np.mean(e))
    return np.nan if r - o < 1e-12 else (f(s) - o) / (r - o)

print("1. 시험 분자별 위험 점수 추출 및 군집 정보 병합 중...")
datasets_data = {}

for d in sorted(p.name for p in EV.iterdir() if p.is_dir() and not p.name.startswith("_")):
    m = pd.read_csv(EV / d / "evaluation_signals.csv")
    me, te = m[m.split == "meta"], m[m.split == "test"]

    # splits.csv에서 scaffold_group 정보 가져와서 합치기
    splits_path = SPLITS_DIR / d / "splits.csv"
    if splits_path.exists():
        splits_df = pd.read_csv(splits_path, low_memory=False)
        te = pd.merge(te, splits_df[['row_uid', 'scaffold_group']], on='row_uid', how='left')
    else:
        te['scaffold_group'] = np.arange(len(te)) # 파일이 없을 경우 대비 안전장치

    err = te.abs_error_fp.to_numpy(float)
    tgt = rankdata(me.abs_error_fp) / len(me)

    scores = {}
    for n, f in CFG.items():
        u = [c for c in f if c in m and m[c].nunique() > 1]
        s = Ridge(alpha=1.0).fit(me[u].to_numpy(float), tgt).predict(te[u].to_numpy(float))
        scores[n] = s

    df_test = pd.DataFrame({
        'row_uid': te['row_uid'].values if 'row_uid' in te else np.arange(len(te)),
        'scaffold_group': te['scaffold_group'].values,
        'err': err,
        'score_base': scores["기준(AD+컨포멀)"],
        'score_base_B': scores["기준+B"]
    })
    datasets_data[d] = df_test

print(f"총 {len(datasets_data)}개 물성에 대한 예측 완료.")
print("\n2. 골격 군집 단위 부트스트랩 2000회 진행 중... (시간이 다소 소요됩니다)")

N_BOOTSTRAP = 2000
boot_mean_improvements = []

for i in range(N_BOOTSTRAP):
    if (i + 1) % 500 == 0:
        print(f" - Bootstrap 진행 상황: {i + 1} / {N_BOOTSTRAP}")

    improvements_in_iter = []
    for dataset_name, df in datasets_data.items():
        df['scaffold_group'] = df['scaffold_group'].fillna('unknown') # 결측치 안전장치
        unique_scaffolds = df['scaffold_group'].unique()
        sampled_scaffolds = np.random.choice(unique_scaffolds, size=len(unique_scaffolds), replace=True)

        scaf_dict = dict(tuple(df.groupby('scaffold_group')))
        sample_df = pd.concat([scaf_dict[scaf] for scaf in sampled_scaffolds], ignore_index=True)

        err_sample = sample_df['err'].values
        s_base = sample_df['score_base'].values
        s_base_B = sample_df['score_base_B'].values

        aurc_base = naurc(s_base, err_sample)
        aurc_base_B = naurc(s_base_B, err_sample)

        if not np.isnan(aurc_base) and not np.isnan(aurc_base_B):
            improvements_in_iter.append(aurc_base - aurc_base_B)

    mean_imp = np.mean(improvements_in_iter)
    boot_mean_improvements.append(mean_imp)

boot_mean_improvements = np.array(boot_mean_improvements)
ci_lower = np.percentile(boot_mean_improvements, 2.5)
ci_upper = np.percentile(boot_mean_improvements, 97.5)
p_value = np.sum(boot_mean_improvements <= 0) / N_BOOTSTRAP
real_mean = np.mean(boot_mean_improvements)

print("\n=== 최종 분석 결과 (GitHub 보고서 업로드용) ===")
print(f"22개 물성 평균 개선량 (기준 - 기준+B): {real_mean:.4f}")
print(f"95% 신뢰구간: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"진짜 p-value: {p_value:.4f}")

Overwriting /content/drive/MyDrive/Conference_2026/Youngju/5주차 Task/run_preregistered_ablation.py


In [18]:
!python "/content/drive/MyDrive/Conference_2026/Youngju/5주차 Task/run_preregistered_ablation.py"

1. 시험 분자별 위험 점수 추출 및 군집 정보 병합 중...
총 22개 물성에 대한 예측 완료.

2. 골격 군집 단위 부트스트랩 2000회 진행 중... (시간이 다소 소요됩니다)
 - Bootstrap 진행 상황: 500 / 2000
 - Bootstrap 진행 상황: 1000 / 2000
 - Bootstrap 진행 상황: 1500 / 2000
 - Bootstrap 진행 상황: 2000 / 2000

=== 최종 분석 결과 (GitHub 보고서 업로드용) ===
22개 물성 평균 개선량 (기준 - 기준+B): 0.0266
95% 신뢰구간: [-0.0002, 0.0540]
진짜 p-value: 0.0275


In [19]:
%%writefile "/content/drive/MyDrive/Conference_2026/Youngju/5주차 Task/compute_ab_variance_mad.py"
#!/usr/bin/env python
"""변형 예측에서 A축·B축 분산 신호를 만든다 (연구계획서 4.2·5.8절).

피드백 반영: 기존 표준편차(std) 대신 중위수 절대 편차(MAD)를 계산하도록 수정됨.
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, median_abs_deviation

AXES = ("A", "B1_tautomer", "B1_protonation", "B3_stereo")
B_AXES = ("B1_tautomer", "B1_protonation", "B3_stereo")

MODELS = {
    "fp_primary": "pred_fp_primary",
    "cb_regular": "pred_chemberta_regular",
    "cb_augmented": "pred_chemberta_augmented",
}
TARGET_SPLITS = ("meta", "test")

# 부동소수점 잔차를 0으로 본다.
ZERO_TOLERANCE = 1e-12

def _spread(values: np.ndarray) -> tuple[float, float]:
    """표준편차 대신 MAD와 범위(range)를 반환하도록 수정"""
    if len(values) < 2:
        return 0.0, 0.0

    # 수정: np.std 대신 median_abs_deviation 사용
    mad = float(median_abs_deviation(values, scale='normal'))
    span = float(values.max() - values.min())
    return (
        0.0 if mad < ZERO_TOLERANCE else mad,
        0.0 if span < ZERO_TOLERANCE else span,
    )

def build_signals(dataset: str, splits_dir: Path, scores_dir: Path) -> pd.DataFrame:
    splits = pd.read_csv(splits_dir / dataset / "splits.csv", low_memory=False)
    splits = splits[splits["split"].isin(TARGET_SPLITS)].reset_index(drop=True)

    fp_dir = scores_dir / "fingerprint" / dataset
    cb_dir = scores_dir / "chemberta" / dataset
    origin = (
        pd.read_csv(fp_dir / "origin_predictions_refit.csv")
        .merge(
            pd.read_csv(cb_dir / "origin_predictions_chemberta.csv").drop(
                columns=["dataset", "split"]
            ),
            on="row_uid",
        )
        .set_index("row_uid")
    )
    variants = pd.read_csv(fp_dir / "variant_predictions_fp.csv").merge(
        pd.read_csv(cb_dir / "variant_predictions_chemberta.csv").drop(
            columns=["dataset", "axis", "split", "parent_row_uid"]
        ),
        on="variant_uid",
    )

    rows = []
    grouped = {key: frame for key, frame in variants.groupby("parent_row_uid")}
    for row_uid in splits["row_uid"]:
        record: dict = {"row_uid": row_uid}
        group = grouped.get(row_uid)
        for model_key, column in MODELS.items():
            parent_value = float(origin.at[row_uid, column])
            b_pool = [parent_value]
            for axis in AXES:
                if group is None:
                    axis_values = np.array([], dtype=float)
                else:
                    axis_values = group.loc[group["axis"] == axis, column].to_numpy(
                        dtype=float
                    )
                sample = np.append(axis_values, parent_value)
                mad, span = _spread(sample)
                # 열 이름을 std에서 mad로 모두 수정
                record[f"{model_key}__{axis}__mad"] = mad
                record[f"{model_key}__{axis}__range"] = span
                record[f"{model_key}__{axis}__n"] = int(len(axis_values))
                if axis in B_AXES:
                    b_pool.extend(axis_values.tolist())
            mad, span = _spread(np.asarray(b_pool, dtype=float))
            # 열 이름을 std에서 mad로 수정
            record[f"{model_key}__B_combined__mad"] = mad
            record[f"{model_key}__B_combined__range"] = span
            record[f"{model_key}__point"] = parent_value
        rows.append(record)

    signals = pd.DataFrame(rows)
    frame = splits[["row_uid", "dataset", "task_type", "split", "cv_fold", "Y_final"]].merge(
        signals, on="row_uid"
    )

    task_type = frame["task_type"].iloc[0]
    truth = pd.to_numeric(frame["Y_final"]).to_numpy(dtype=float)
    for model_key in MODELS:
        frame[f"{model_key}__abs_error"] = np.abs(
            truth - frame[f"{model_key}__point"].to_numpy(dtype=float)
        )
    frame["task_type"] = task_type

    # 백분위 정규화 대상 열도 __mad로 수정
    for column in [c for c in frame.columns if c.endswith("__mad")]:
        frame[f"{column}_pct"] = frame[column].rank(pct=True, method="average")
    return frame

def summarize(frame: pd.DataFrame) -> list[dict]:
    subset = frame[frame["split"].eq("test")]
    out = []
    for model_key in MODELS:
        error = subset[f"{model_key}__abs_error"].to_numpy(dtype=float)
        for axis in (*AXES, "B_combined"):
            # std 대신 mad 열 참조
            signal = subset[f"{model_key}__{axis}__mad"].to_numpy(dtype=float)
            if np.allclose(signal, signal[0]) or np.allclose(error, error[0]):
                rho = np.nan
            else:
                rho = float(spearmanr(signal, error).statistic)
            out.append(
                {
                    "dataset": subset["dataset"].iloc[0],
                    "task_type": subset["task_type"].iloc[0],
                    "model": model_key,
                    "axis": axis,
                    "n_test": len(subset),
                    "signal_mean": float(signal.mean()),
                    "signal_nonzero_rate": float((signal > 0).mean()),
                    "spearman_vs_abs_error": rho,
                }
            )
    return out

def main() -> int:
    parser = argparse.ArgumentParser(description="A축·B축 분산 신호 산출 (MAD 적용)")
    parser.add_argument("--splits-dir", required=True)
    parser.add_argument("--scores-dir", required=True)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--datasets", nargs="*", default=None)
    args = parser.parse_args()

    splits_dir = Path(args.splits_dir)
    scores_dir = Path(args.scores_dir)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = args.datasets or sorted(
        path.name
        for path in (scores_dir / "fingerprint").iterdir()
        if path.is_dir() and not path.name.startswith("_")
    )

    all_summaries = []
    for index, dataset in enumerate(datasets, 1):
        frame = build_signals(dataset, splits_dir, scores_dir)
        dataset_out = out_dir / dataset
        dataset_out.mkdir(parents=True, exist_ok=True)
        frame.to_csv(dataset_out / "ab_signals.csv", index=False)
        all_summaries.extend(summarize(frame))
        print(f"[{index}/{len(datasets)}] {dataset}: {len(frame):,}행", flush=True)

    summary = pd.DataFrame(all_summaries)
    summary_dir = out_dir / "_summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(summary_dir / "ab_signal_summary.csv", index=False)

    print()
    print("=== 축·모델별 신호 크기(MAD)와 오차 상관 (test, 22종 중앙값) ===")
    pivot = (
        summary.groupby(["model", "axis"])
        .agg(
            신호평균=("signal_mean", "median"),
            비영비율=("signal_nonzero_rate", "median"),
            상관중앙=("spearman_vs_abs_error", "median"),
            상관양수=("spearman_vs_abs_error", lambda s: int((s > 0).sum())),
        )
        .reset_index()
    )
    print(pivot.to_string(index=False))
    return 0

if __name__ == "__main__":
    raise SystemExit(main())

Overwriting /content/drive/MyDrive/Conference_2026/Youngju/5주차 Task/compute_ab_variance_mad.py


In [20]:
!python "/content/drive/MyDrive/Conference_2026/Youngju/5주차 Task/compute_ab_variance_mad.py" \
--splits-dir "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo" \
--scores-dir "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/scores_role4" \
--out-dir "/content/drive/MyDrive/Conference_2026/Youngju/signals_mad"

[1/22] ames: 1,450행
[2/22] bbb_martins: 390행
[3/22] bioavailability_ma: 127행
[4/22] caco2_wang: 180행
[5/22] clearance_hepatocyte_az: 203행
[6/22] clearance_microsome_az: 220행
[7/22] cyp2c9_substrate_carbonmangels: 133행
[8/22] cyp2c9_veith: 2,409행
[9/22] cyp2d6_substrate_carbonmangels: 132행
[10/22] cyp2d6_veith: 2,616행
[11/22] cyp3a4_substrate_carbonmangels: 133행
[12/22] cyp3a4_veith: 2,460행
[13/22] dili: 94행
[14/22] half_life_obach: 132행
[15/22] herg: 127행
[16/22] hia_hou: 116행
[17/22] ld50_zhu: 1,468행
[18/22] lipophilicity_astrazeneca: 838행
[19/22] pgp_broccatelli: 242행
[20/22] ppbr_az: 358행
[21/22] solubility_aqsoldb: 1,915행
[22/22] vdss_lombardo: 222행

=== 축·모델별 신호 크기(MAD)와 오차 상관 (test, 22종 중앙값) ===
       model           axis     신호평균     비영비율     상관중앙  상관양수
cb_augmented              A 0.046590 1.000000 0.141324    19
cb_augmented B1_protonation 0.005912 0.286797 0.029114    16
cb_augmented    B1_tautomer 0.017818 0.522727 0.039513    14
cb_augmented      B3_stereo 0.001101 0.087524